In [44]:
import pandas as pd

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [46]:
#Load the CSV

data=pd.read_csv("./Dataset/Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [47]:
#Pre-process the data

#We wont require RowNumber,CustomerID and surname to train our data since they dont hold much value
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [48]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [49]:
#Encoding the Gender

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [50]:
#One Hot encode Geography Column
#We dont use Label Encoder since there are more than 2 categories and no Geography should have numerical
#superiority other than 0 and 1

from sklearn.preprocessing import OneHotEncoder

one_hot_encoder_geo=OneHotEncoder()

In [ ]:
# Why One-Hot Encoding here (and not BoW / TF-IDF)?
# ---------------------------------------------------
# 1. Nature of the feature:
#    'Geography' is a CATEGORICAL feature with a small, fixed set of
#    nominal values (e.g., France, Germany, Spain). It is NOT free text.
#    One-Hot Encoding is the standard way to represent such nominal
#    categories for ML/DL models because:
#      - It avoids implying any ordinal relationship (unlike Label Encoding).
#      - Each category becomes an independent binary feature.
#
# 2. Why NOT Bag-of-Words (BoW)?
#    BoW is designed for TEXT DOCUMENTS — it counts how often each word
#    appears in a document. Here, every row has exactly ONE token
#    ("France", "Germany", "Spain"), so BoW would essentially collapse
#    into the same representation as One-Hot Encoding, just via a more
#    expensive text-processing pipeline. No benefit.
#
# 3. Why NOT TF-IDF?
#    TF-IDF weights words by how rare they are across documents. Since
#    each row has only one token, TF=1 for that token and 0 elsewhere,
#    and the IDF term just rescales columns by a constant. The result is
#    a scaled one-hot vector — no extra information gained, and the
#    scaling can actually hurt model interpretability.
#
# 4. When WOULD BoW / TF-IDF make sense?
#    Only if the column contained free-form text with multiple words per
#    row (e.g., customer reviews, product descriptions, complaints).
#
# Conclusion: For a single-token categorical column like 'Geography',
# One-Hot Encoding is the correct and most efficient choice.
geo_encoder=one_hot_encoder_geo.fit_transform(data[['Geography']])

In [52]:
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [53]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [54]:
#Convert geo_encoder array to Dataframe to fit in our original dataframe (Dont yet fit)
encoded_df = pd.DataFrame(geo_encoder,columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

ValueError: Shape of passed values is (10000, 1), indices imply (10000, 3)

In [55]:
## The above error occurs because each we are passing the encoded object, not the array

# TO do it ... Pass the geo_encoded .toarray()

encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

In [56]:
encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [57]:
data = pd.concat([data.drop('Geography', axis=1), encoded_df], axis=1)

In [58]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [59]:
#Save the encoders and scaler for later uses

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

In [60]:
#Split the data into dependent and independent features

#Exit is dependent , while others are independent 

X=data.drop('Exited',axis=1) # Indepndent --> All other features other than exited
y=data['Exited'] #Dependent


#Split the data into training and testing datasets

# random_state=42 ensures reproducibility: the same train/test split is produced every run.
# test_size=0.2 means 20% of the data is reserved for testing, 80% for training.
# Note on random_state=42:
# - The number 42 has NO mathematical significance. It's a popular convention (42 has no meaning)

# - Its only purpose is to SEED the random number generator so that the
#   train/test split is reproducible — you get the exact same split every run.
# - You could use ANY integer (0, 1, 7, 100, 12345, etc.) and it would work
#   equally well. The model's quality does NOT depend on this number.
# - What changes if you lower/raise it?
#     * Different seed  -> different random shuffle -> different rows go into
#       train vs test -> slightly different accuracy/loss numbers.
#     * The underlying data, model, and learning process are unchanged.
#     * If you set random_state=None, you get a NEW random split every run
#       (results won't be reproducible).
# - Best practice: pick any fixed integer and keep it constant while you
#   experiment, so performance changes reflect your code/model changes —
#   not random luck in the split.
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


#Scale the data

scaler = StandardScaler()

# Why scale the data?
# Neural networks (and many ML algorithms) work better when input features are on a similar scale.
# Reasons for scaling:
# 1. Faster convergence: Gradient descent converges much faster when features have similar ranges.
#    Without scaling, features with larger magnitudes dominate the gradient updates.
# 2. Prevents bias toward large-valued features: e.g., 'EstimatedSalary' (in 100,000s) would
#    overshadow 'Age' (in 10s) if not scaled, even though both may be equally important.
# 3. Activation functions (sigmoid, tanh) saturate for large input values, causing vanishing
#    gradients. Scaling keeps inputs in a range where activations are sensitive.
# 4. Weight initialization assumptions: Most initialization schemes (Xavier, He) assume
#    inputs are roughly zero-mean and unit-variance.
#
# StandardScaler transforms data to have mean=0 and standard deviation=1 (z-score normalization).
# Note: We use fit_transform on training data, but should use only transform on test data
# to avoid data leakage (test set should be scaled using training set's statistics).
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [ ]:
# X_train is a NumPy array (not a DataFrame) because of how sklearn's
# train_test_split works internally. Here's the conversion chain:
#
# 1. Originally, your data was a pandas DataFrame (e.g., df).
# 2. When you applied transformations like StandardScaler.fit_transform(X)
#    or ColumnTransformer/Pipeline, sklearn ALWAYS returns a NumPy ndarray
#    (or sparse matrix), NOT a DataFrame. This is the main converter.
# 3. train_test_split then just splits that ndarray into X_train / X_test,
#    preserving the ndarray type.
#
# So the "table -> array" conversion happens at the sklearn preprocessing
# step (StandardScaler / OneHotEncoder / ColumnTransformer), not at split.
#
# Are they vectors?
# - X_train is a 2D array of shape (n_samples, n_features) -> a MATRIX.
#   Each ROW is a feature vector for one sample (customer).
# - y_train is a 1D array of shape (n_samples,) -> a VECTOR of labels.
#
# You can verify with:
#   type(X_train), X_train.shape, X_train.ndim
#
# If you want it back as a DataFrame, do:
#   pd.DataFrame(X_train, columns=feature_names)

X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [62]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [63]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


## ANN Implementation

1. Create Sequential Model
2. Create Nodes (with Dense library of Keras)
3. Activation Functions (like ReLU, Sigmoid, etc.)
4. Use Optimizer (like Adam, SGD, etc.) and Back Propagation for updating the weights
5. Calculate Loss Function (like Mean Squared Error, Cross Entropy, etc.) and try to minimize it during training
6. Evaluate the model's performance on the validation/test dataset

In [66]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
## Build our Model(Step 1 and 2)

# ============================================================
# UNDERSTANDING THE ANN ARCHITECTURE
# ============================================================
# Sequential -> A linear stack of layers (data flows top-down,
#               one layer feeds into the next).
#
# Why 64 nodes (neurons) in the first hidden layer?
#   - 64 is a HYPERPARAMETER (a design choice, not learned).
#   - More neurons => more capacity to learn complex patterns,
#     but too many => overfitting + slow training.
#   - Powers of 2 (32, 64, 128) are commonly used because they
#     map well to GPU/CPU memory. 64 is a good starting point
#     for tabular data with ~10-20 input features.
#
# What is the "input"?
#   - The input is one row of X_train (one customer record)
#     containing all the features (CreditScore, Age, Balance,
#     Geography_*, Gender_*, etc. after encoding/scaling).
#   - The network does NOT take the target column y.
#
# Why input_shape=(X_train.shape[1],) ?
#   - X_train is a 2D array of shape (num_samples, num_features).
#   - X_train.shape[0] = number of rows (samples).
#   - X_train.shape[1] = number of columns (features per sample).
#   - Keras only needs to know the shape of ONE sample, so we
#     pass (num_fe9atures,) as a tuple. The batch dimension is
#     handled automatically.
#   - Using X_train.shape[1] makes the code generic: if you add
#     or remove features, you don't have to hardcode the number.
#
# Flow summary:
#   Input (n_features) -> Dense(64, relu) -> Dense(32, relu)
#                      -> Dense(1, sigmoid) -> probability (0-1)
# ============================================================
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## Hidden Layer 1 Connected to input layer
    Dense(32,activation='relu'), #Hidden-Layer2
    Dense(1,activation='sigmoid') #Output Layer

])

In [71]:
# Initialize the Optimizers and Loss Functions

opt_function=tf.keras.optimizers.Adam(learning_rate=0.01)
loss_function= tf.keras.losses.BinaryCrossentropy()


In [72]:
#Compile the Model

model.compile(optimizer=opt_function,loss=loss_function,metrics=['accuracy'])

In [77]:
# Setup the tensor board , helps keeping logs efficienctly

logs_directory="logs/fit" + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

tensor_flow_callback=TensorBoard(log_dir=logs_directory,histogram_freq=1)


In [80]:
#Early Stopping Configuration

# Say after 5 epochs (Epoch= Each pass of training dataset), the loss value is not decreasing and remains constant
#Then we dont need to run more epoch (Say 100s) further because that is the lowest Loss value we can get. Stop the training.

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=20,restore_best_weights=True)#  patience = Wait for 5 epochs to 
                                                                                               #notice if loss function is FURTHER GOING DOWN or IT IS CONSTANT
#D restore_best_weight = Restore Best Weights in during constant Loss value

In [81]:
history = model.fit(
X_train,y_train,validation_data=(X_test, y_test),epochs=100,
callbacks=[tensor_flow_callback,early_stopping_callback]

)

Epoch 1/100
  1/250 [..............................] - ETA: 3s - loss: 0.3835 - accuracy: 0.7812

250/250 [==============================] - 1s 3ms/step - loss: 0.3391 - accuracy: 0.8610 - val_loss: 0.3475 - val_accuracy: 0.8565
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3381 - accuracy: 0.8612 - val_loss: 0.3538 - val_accuracy: 0.8595
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3337 - accuracy: 0.8637 - val_loss: 0.3499 - val_accuracy: 0.8620
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3305 - accuracy: 0.8640 - val_loss: 0.3584 - val_accuracy: 0.8530
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3296 - accuracy: 0.8666 - val_loss: 0.3467 - val_accuracy: 0.8620
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3275 - accuracy: 0.8671 - val_loss: 0.3515 - val_accuracy: 0.8645
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3267 - accuracy: 0.8666 - val_loss: 0.3414 - val_accuracy: 0.8550
Epoch 8/100

In [82]:
model.save('model.h5')

c:\avishek_work\Machine Learning Projects\ANN Project\.conda\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [94]:
# Training ended , now lets see the Tensor Board

%reload_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit20260425-171513

In [ ]:

# All requested packages already installed.

In [1]:
#Model is now trained , now lets use it for prediction

## Go to prediction.ipynb